In [1]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt
import sklearn


In [4]:
X=pd.read_csv(r'/Users/marc-antoine/Desktop/3A/CS/Machine Learning B/TP 1/bardet_x.csv')
y=pd.read_csv(r'/Users/marc-antoine/Desktop/3A/CS/Machine Learning B/TP 1/bardet_y.csv')

In [ ]:
#Ridge regression with cross-validation
from sklearn import linear_model
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV
lambdas = np.logspace(-4, 4, 100)
gridsearch = GridSearchCV(Ridge(),param_grid={'alpha':lambdas},cv=5,scoring='neg_mean_squared_error')
gridsearch.fit(X,y.values.ravel())
mse_mean = -gridsearch.cv_results_['mean_test_score']
mse_std = gridsearch.cv_results_['std_test_score']
n_folds = gridsearch.n_splits_
sem = mse_std / np.sqrt(n_folds)
#Plot of the cross validation error as a function of the regularization parameter
#And the best value of the regularization parameter
plt.figure(figsize=(8,6))
plt.semilogx(lambdas, mse_mean, label='Mean CV Error')
plt.fill_between(lambdas, mse_mean - sem, mse_mean + sem, alpha=0.2, label='± SEM')
plt.axvline(gridsearch.best_params_['alpha'], color='r', linestyle='--', label='Best lambda')
plt.xlabel('Regularization parameter (lambda)')
plt.ylabel('Mean Squared Error')
plt.title('Ridge Regression Cross-Validation Error')
plt.legend()
plt.show()


In [ ]:
#Same with lasso regression
from sklearn.linear_model import Lasso
gridsearch_lasso = GridSearchCV(Lasso(max_iter=10000),param_grid={'alpha':lambdas},cv=5,scoring='neg_mean_squared_error')
gridsearch_lasso.fit(X,y.values.ravel())
mse_mean_lasso = -gridsearch_lasso.cv_results_['mean_test_score']
mse_std_lasso = gridsearch_lasso.cv_results_['std_test_score']
n_folds_lasso = gridsearch_lasso.n_splits_
sem_lasso = mse_std_lasso / np.sqrt(n_folds_lasso)
#Plot of the cross validation error as a function of the regularization parameter
#And the best value of the regularization parameter
plt.figure(figsize=(8,6))
plt.semilogx(lambdas, mse_mean_lasso, label='Mean CV Error')
plt.fill_between(lambdas, mse_mean_lasso - sem_lasso, mse_mean_lasso + sem_lasso, alpha=0.2, label='± SEM')
plt.axvline(gridsearch_lasso.best_params_['alpha'], color='r', linestyle='--', label='Best lambda')
plt.xlabel('Regularization parameter (lambda)')
plt.ylabel('Mean Squared Error')
plt.title('Lasso Regression Cross-Validation Error')
plt.legend()
plt.show()

In [ ]:
#Pareil avec elastic net regression
from sklearn.linear_model import ElasticNet
l1_ratios = np.linspace(0.01, 1, 20)  # l1_ratio doit rester dans [0, 1] (0=ridge, 1=lasso)
gridsearch_elastic = GridSearchCV(
    ElasticNet(max_iter=10000),
    param_grid={'alpha': lambdas, 'l1_ratio': l1_ratios},
    cv=5,
    scoring='neg_mean_squared_error'
)
gridsearch_elastic.fit(X, y.values.ravel())

# alpha varie plus lentement que l1_ratio dans cv_results_ (ordre alphabétique des clés)
mse_grid_elastic = -gridsearch_elastic.cv_results_['mean_test_score'].reshape(len(lambdas), len(l1_ratios))
std_grid_elastic = gridsearch_elastic.cv_results_['std_test_score'].reshape(len(lambdas), len(l1_ratios))

best_alpha = gridsearch_elastic.best_params_['alpha']
best_l1 = gridsearch_elastic.best_params_['l1_ratio']
best_l1_idx = np.argmin(np.abs(l1_ratios - best_l1))

mse_mean_elastic = mse_grid_elastic[:, best_l1_idx]
sem_elastic = std_grid_elastic[:, best_l1_idx] / np.sqrt(gridsearch_elastic.n_splits_)

#Plot of the cross validation error as a function of alpha, à l1_ratio optimal fixé
plt.figure(figsize=(8,6))
plt.semilogx(lambdas, mse_mean_elastic, label=f'Mean CV Error (l1_ratio={best_l1:.2f})')
plt.fill_between(lambdas, mse_mean_elastic - sem_elastic, mse_mean_elastic + sem_elastic, alpha=0.2, label='± SEM')
plt.axvline(best_alpha, color='r', linestyle='--', label='Best lambda')
plt.xlabel('Regularization parameter (lambda)')
plt.ylabel('Mean Squared Error')
plt.title('Elastic Net Regression Cross-Validation Error')
plt.legend()
plt.show()

In [ ]:
#Visualisation des groupes de variables (gènes) sélectionnés par elastic net
best_elastic = gridsearch_elastic.best_estimator_
coefs = best_elastic.coef_
coefs_grouped = coefs.reshape(20, 5)  # 20 gènes candidats x 5 fonctions de base B-splines

# Heatmap des coefficients par gène / fonction de base
vmax = np.abs(coefs).max()
plt.figure(figsize=(10, 6))
plt.imshow(coefs_grouped, aspect='auto', cmap='RdBu_r', vmin=-vmax, vmax=vmax)
plt.colorbar(label='Coefficient')
plt.xlabel('B-spline basis function (1 to 5)')
plt.ylabel('Candidate gene')
plt.yticks(range(20), [f'Gene {i+1}' for i in range(20)])
plt.xticks(range(5), [f'{j+1}' for j in range(5)])
plt.title('Elastic Net coefficients by gene and basis function')
plt.show()

# Nombre de coefficients non-nuls par gène : sélection complète (5/5), partielle, ou nulle
n_nonzero_per_gene = np.sum(coefs_grouped != 0, axis=1)
colors = ['#2c7fb8' if n == 5 else '#d9d9d9' if n == 0 else '#f16913' for n in n_nonzero_per_gene]

plt.figure(figsize=(10, 6))
plt.bar(range(1, 21), n_nonzero_per_gene, color=colors)
plt.axhline(5, color='k', linestyle='--', linewidth=1, label='Full group (5/5)')
plt.xlabel('Candidate gene')
plt.ylabel('Number of non-zero coefficients (out of 5)')
plt.xticks(range(1, 21))
plt.title('Gene-level selection — elastic net (blue = full group, orange = partial selection, gray = not selected)')
plt.legend()
plt.show()

print(f"Gènes non retenus (0/5) : {np.sum(n_nonzero_per_gene == 0)}")
print(f"Gènes partiellement retenus (1 à 4 sur 5) : {np.sum((n_nonzero_per_gene > 0) & (n_nonzero_per_gene < 5))}")
print(f"Gènes entièrement retenus (5/5) : {np.sum(n_nonzero_per_gene == 5)}")


In [ ]:
#Group Lasso (pip install group-lasso), avec 20 groupes de taille pg=5
from group_lasso import GroupLasso
groups = np.repeat(np.arange(20), 5)  # groupes[i] = indice du gène (0 à 19) de la colonne i

lambdas_group = np.logspace(-4, -1, 50)  # échelle différente de "lambdas" (group_reg a une autre plage utile)
gl = GroupLasso(groups=groups, l1_reg=0, n_iter=5000, supress_warning=True)  # l1_reg=0 -> group Lasso pur (pas sparse group lasso)

gridsearch_group = GridSearchCV(gl, param_grid={'group_reg': lambdas_group}, cv=5, scoring='neg_mean_squared_error')
gridsearch_group.fit(X, y.values.ravel())

mse_mean_group = -gridsearch_group.cv_results_['mean_test_score']
mse_std_group = gridsearch_group.cv_results_['std_test_score']
sem_group = mse_std_group / np.sqrt(gridsearch_group.n_splits_)

plt.figure(figsize=(8,6))
plt.semilogx(lambdas_group, mse_mean_group, label='Mean CV Error')
plt.fill_between(lambdas_group, mse_mean_group - sem_group, mse_mean_group + sem_group, alpha=0.2, label='± SEM')
plt.axvline(gridsearch_group.best_params_['group_reg'], color='r', linestyle='--', label='Best lambda')
plt.xlabel('Regularization parameter (lambda)')
plt.ylabel('Mean Squared Error')
plt.title('Group Lasso Regression Cross-Validation Error')
plt.legend()
plt.show()

# Gènes retenus par le group Lasso (groupe entier sélectionné ou non, par construction)
best_group_model = gridsearch_group.best_estimator_
coefs_group = best_group_model.coef_.ravel()
coefs_group_grouped = coefs_group.reshape(20, 5)
genes_selected_group = np.where(np.any(coefs_group_grouped != 0, axis=1))[0] + 1
print(f"Gènes candidats retenus par le group Lasso : {genes_selected_group}")
print(f"Nombre de gènes retenus : {len(genes_selected_group)} / 20")

# Comparaison visuelle avec le pattern (potentiellement partiel) de l'elastic net
vmax = np.abs(coefs_group).max()
plt.figure(figsize=(10, 6))
plt.imshow(coefs_group_grouped, aspect='auto', cmap='RdBu_r', vmin=-vmax, vmax=vmax)
plt.colorbar(label='Coefficient')
plt.xlabel('B-spline basis function (1 to 5)')
plt.ylabel('Candidate gene')
plt.yticks(range(20), [f'Gene {i+1}' for i in range(20)])
plt.xticks(range(5), [f'{j+1}' for j in range(5)])
plt.title('Group Lasso coefficients by gene and basis function')
plt.show()

# Nombre de coefficients non-nuls par gène : doit être 0 ou 5 (jamais entre les deux) pour le group Lasso
n_nonzero_per_gene_group = np.sum(coefs_group_grouped != 0, axis=1)
colors_group = ['#2c7fb8' if n == 5 else '#d9d9d9' if n == 0 else '#f16913' for n in n_nonzero_per_gene_group]

plt.figure(figsize=(10, 6))
plt.bar(range(1, 21), n_nonzero_per_gene_group, color=colors_group)
plt.axhline(5, color='k', linestyle='--', linewidth=1, label='Full group (5/5)')
plt.xlabel('Candidate gene')
plt.ylabel('Number of non-zero coefficients (out of 5)')
plt.xticks(range(1, 21))
plt.title('Gene-level selection — group Lasso (all-or-nothing per group, no orange bars expected)')
plt.legend()
plt.show()
